## 手写GRU
#### 更新门
$ z_t=sigmoid(W_xz \cdot x_t+W_hz \cdot h_{t-1}+b_z) $
#### 重置门
$ r_t=sigmoid(W_xr \cdot x_t+W_hr \cdot h_{t-1}+b_r) $
#### 候选隐状态
$ h_t=tanh(W_xh \cdot x_t+W_hh \cdot (r_t \cdot h_{t-1})+b_h) $
#### 输出门
$ h_t=z_t \cdot h_t+(1-z_t) \cdot h_{t-1} $

In [ ]:
from torch import nn
import torch


class GRUCell(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size

        self.W_xz = nn.Parameter(torch.randn(input_size, hidden_size))
        self.W_hz = nn.Parameter(torch.randn(hidden_size, hidden_size))

        self.W_xr = nn.Parameter(torch.randn(input_size, hidden_size))
        self.W_hr = nn.Parameter(torch.randn(hidden_size, hidden_size))

        self.W_xh = nn.Parameter(torch.randn(input_size, hidden_size))
        self.W_hh = nn.Parameter(torch.randn(hidden_size, hidden_size))

        self.b_z = nn.Parameter(torch.zeros(hidden_size))
        self.b_r = nn.Parameter(torch.zeros(hidden_size))
        self.b_h = nn.Parameter(torch.zeros(hidden_size))

    def forward(self, x, h_prev):
        # 更新门
        z_t = torch.sigmoid(torch.mm(x, self.W_xz) + torch.mm(h_prev, self.W_hz) + self.b_z)
        # 重置门
        r_t = torch.sigmoid(torch.mm(x, self.W_xr) + torch.mm(h_prev, self.W_hr) + self.b_r)
        # 候选隐状态
        h_candidate = torch.tanh(torch.mm(x, self.W_xh) + torch.mm(r_t * h_prev, self.W_hh) + self.b_h)
        # 最终隐状态
        h_t = z_t * h_candidate + (1 - z_t) * h_prev
        return h_t


class GRU(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.cell = GRUCell(input_size, hidden_size)

    def forward(self, x, hidden=None):
        seq_len, batch_size, input_size = x.shape
        if hidden is None:
            hidden = torch.zeros(batch_size, self.hidden_size)
        outputs = []
        for t in range(seq_len):
            hidden = self.cell(x[t], hidden)
            outputs.append(hidden)
        return outputs, hidden





In [ ]:
input_size = 32
hidden_size = 5
batch_size = 1
seq_len = 2


model = GRU(input_size, hidden_size)
x = torch.randn(seq_len, batch_size, input_size)
outputs, hidden = model(x)

print(outputs)
print(hidden)


## GRU文本生成

In [7]:
text = """
臣密言：臣以险衅，夙遭闵凶。生孩六月，慈父见背；行年四岁，舅夺母志。
祖母刘愍臣孤弱，躬亲抚养。
臣少多疾病，九岁不行，零丁孤苦，至于成立。
既无伯叔，终鲜兄弟，门衰祚薄，晚有儿息。
外无期功强近之亲，内无应门五尺之僮，茕茕孑立，形影相吊。
而刘夙婴疾病，常在床蓐，臣侍汤药，未曾废离。
"""

words = set(text)
vocab_size = len(words)
word_to_index = {word: i for i, word in enumerate(words)}
index_to_word = {i: word for i, word in enumerate(words)}

print(word_to_index)

{'弱': 0, '孑': 1, '蓐': 2, '影': 3, '近': 4, '伯': 5, '养': 6, '六': 7, '汤': 8, '苦': 9, '薄': 10, '慈': 11, '\n': 12, '兄': 13, '零': 14, '背': 15, '父': 16, '。': 17, '门': 18, '无': 19, '应': 20, '母': 21, '相': 22, '吊': 23, '尺': 24, '岁': 25, '抚': 26, '至': 27, '疾': 28, '强': 29, '言': 30, '：': 31, '在': 32, '孤': 33, '孩': 34, '凶': 35, '生': 36, '四': 37, '愍': 38, '年': 39, '曾': 40, '祚': 41, '侍': 42, '遭': 43, '弟': 44, '九': 45, '祖': 46, '不': 47, '婴': 48, '常': 49, '以': 50, '叔': 51, '密': 52, '见': 53, '儿': 54, '未': 55, '鲜': 56, '床': 57, '离': 58, '丁': 59, '于': 60, '臣': 61, '既': 62, '期': 63, '立': 64, '成': 65, '闵': 66, '药': 67, '刘': 68, '；': 69, '僮': 70, '终': 71, '志': 72, '五': 73, '行': 74, '少': 75, '月': 76, '病': 77, '晚': 78, '内': 79, '之': 80, '衰': 81, '躬': 82, '茕': 83, '息': 84, '而': 85, '，': 86, '外': 87, '舅': 88, '亲': 89, '险': 90, '多': 91, '有': 92, '夺': 93, '衅': 94, '功': 95, '夙': 96, '形': 97, '废': 98}
